# Convert the Bucks intent model to `.litertlm`
Runtime: **CPU is enough** (needs ~5 GB RAM; Colab gives 12 GB). Run all cells. Cell 2 asks you to pick `bucks-intent-checkpoint.zip`; the last cell downloads `bucks-intent_q8_ekv1024.litertlm` (~280 MB).

In [ ]:
# 1. Pinned tools (jax 0.12.1 breaks the export; litert-torch needs torch < 2.15 and imports tensorflow)
!pip install -q "torch<2.15" --index-url https://download.pytorch.org/whl/cpu
!pip install -q "litert-torch==0.10.0" tensorflow-cpu "jax==0.12.0.1" "jaxlib==0.12.0.1"
!pip list 2>/dev/null | grep -E "^(torch|litert-torch|jax|jaxlib|tensorflow-cpu) "

In [ ]:
# 2. Upload the checkpoint zip
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall('ckpt')
print(sorted(os.listdir('ckpt')))

In [ ]:
%%writefile convert_litertlm.py
import glob
import sys

step, ck, out = sys.argv[1], sys.argv[2], sys.argv[3]
Q8 = f"{out}/bucks-intent_q8_ekv1024.tflite"
if step == "export":
    from litert_torch.generative.examples.gemma3 import gemma3
    from litert_torch.generative.layers import kv_cache
    from litert_torch.generative.utilities import converter
    from litert_torch.generative.utilities.export_config import ExportConfig
    cfg = ExportConfig(); cfg.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED; cfg.mask_as_input = True
    print("exported", converter.convert_to_tflite(gemma3.build_model_270m(ck), out, "bucks-intent", 256, 1024,
                                                  quantize="none", export_config=cfg))
elif step == "quantize":
    from litert_torch.generative.examples.gemma3 import decoder
    from litert_torch.generative.quantize import quant_recipes
    from litert_torch.quantize import translate_recipe
    src = glob.glob(f"{out}/bucks-intent_*f32*.tflite")[0]
    recipe = translate_recipe.translate_to_ai_edge_recipe(quant_recipes.full_dynamic_recipe(mcfg=decoder.get_decoder_config_270m()))
    q = translate_recipe.quantize_model(open(src, "rb").read(), recipe)
    open(Q8, "wb").write(q); print("quantized", len(q))
elif step == "pack":
    import tempfile
    from litert_torch.generative.utilities import litertlm_builder
    with tempfile.TemporaryDirectory() as wd:
        litertlm_builder.build_litertlm(
            tflite_model_path=Q8, workdir=wd, output_path=out, context_length=1024,
            hf_tokenizer_model_path=f"{ck}/tokenizer.json", llm_model_type="generic",
            start_token="<bos>", stop_tokens=["<end_of_turn>", "<start_function_response>", "<eos>"],
            user_prompt_prefix="<start_of_turn>user\n", user_prompt_suffix="<end_of_turn>\n",
            model_prompt_prefix="<start_of_turn>model\n", model_prompt_suffix="<end_of_turn>\n",
            jinja_prompt_template=open(f"{ck}/chat_template.jinja").read())
    print("packed")


In [ ]:
# 3. Export, quantize to int8, pack (each step in its own process)
!mkdir -p litertlm
!for s in export quantize pack; do python convert_litertlm.py $s ckpt litertlm 2>&1 | grep -E "exported|quantized|packed|Error|FAIL" || true; done
!ls -la litertlm

In [ ]:
# 4. Download
files.download('litertlm/bucks-intent_q8_ekv1024.litertlm')